# Chapter 19: Hierarchical Parent-Child Retrieval

Parent/child manifests with exact offsets; child-first expansion, parent-first, windows under budgets; useful parents may exceed the cap.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if str(ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(ROOT / "examples"))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from document_identity import Collection, digest_bytes
from evidence_units import evidence_success, version_key
from query_retrieval import Retrieval, interpret, reformulate, Budget
import json
from pathlib import Path

fixture = json.loads((ROOT / "experiments" / "part4-complex" / "development-corpus.json").read_text())
relations = json.loads((ROOT / "experiments" / "part4-complex" / "development-relations.json").read_text())
config = json.loads((ROOT / "experiments" / "part4-complex" / "config.json").read_text())
needs = json.loads((ROOT / "experiments" / "part4-complex" / "development-queries.json").read_text())["queries"]

def engine(view="main"):
    return Retrieval(Collection(fixture), view, relations, config)


In [ ]:
# Hierarchy links require exact same-version containment.
e = engine()
print("number of parent links:", len(e.parents))
for link in e.parents[:5]:
    child = e.units[link["child"]["passage_id"]]
    parent = e.units[link["parent"]["passage_id"]]
    assert version_key(child) == version_key(parent)
    assert child["content_sha256"] == parent["content_sha256"]
    assert parent["span"][0] <= child["span"][0] < child["span"][1] <= parent["span"][1]
print("sample link:", e.parents[0]["child"]["passage_id"], "->", e.parents[0]["parent"]["passage_id"], e.parents[0]["level"])


In [ ]:
# Parent expansion can restore a heading-and-action bundle that child hits alone miss.
q = next(x for x in needs if x["query_id"] == "development-09")
e = engine()
p = e.plan(q["query"], q["query_id"])
out = e.execute(p, "parent-expansion")
print("selected:", [(c["unit"]["document_id"], c["unit"]["passage_id"]) for c in out["selection"]["selected"]])
print("chars:", out["selection"]["characters"], "bundle:", evidence_success(q, [c["unit"] for c in out["selection"]["selected"]]))
assert evidence_success(q, [c["unit"] for c in out["selection"]["selected"]])["bundle_covered"] is True


In [ ]:
# An oversized parent is skipped rather than clipped.
q = next(x for x in needs if x["query_id"] == "development-10")
e = engine()
p = e.plan(q["query"], q["query_id"])
out = e.execute(p, "parent-expansion")
print("selected docs:", [c["unit"]["document_id"] for c in out["selection"]["selected"]])
print("chars:", out["selection"]["characters"])
assert out["selection"]["characters"] <= 900
assert all(c["unit"]["document_id"] != "long" for c in out["selection"]["selected"])


In [ ]:
# Document-first and section-first keep child search inside chosen parents.
e = engine()
p = e.plan("DV-41 service reset", "q")
for mode in ("document-first", "section-first"):
    out = e.execute(p, mode)
    stages = out["cost"]["stages"]
    print(mode, "stages:", len(stages), "candidates:", len(out["candidates"]), "chars:", out["selection"]["characters"])
    assert len(stages) == 2
    assert out["selection"]["characters"] <= 900


In [ ]:
# Restricted ancestors remove all descendants from the unit manifest.
e = engine("ancestor-restricted")
print("mode docs present:", any(u["document_id"] == "mode" for u in e.units.values()))
assert not any(u["document_id"] == "mode" for u in e.units.values())


In [ ]:
# All lexical navigation modes obey the final content cap.
e = engine()
p = e.plan("DV-41 service reset", "q")
for mode in ("lexical", "rewritten", "multi-query", "blocks", "windows12-overlap4", "sections", "document-first", "section-first", "parent-expansion"):
    out = e.execute(p, mode)
    assert out["selection"]["characters"] <= 900
    assert len(out["selection"]["selected"]) <= 4
    assert out["cost"]["candidate_rows"] <= 6
    assert out["cost"]["calls"] <= 2
print("all navigation modes within caps")


## Observed versus not established
Observed: exact parent/child containment, expansion recovery, oversized-parent skipping, scoped child search, and budget compliance. Not established: optimal hierarchy depth, universal parent usefulness, or downstream helpfulness.